# APEX AI — Deep Learning Models
**PyTorch Neural Networks for Fitness Prediction**

This notebook trains deep learning models using PyTorch:
- `FitnessNet` — Multi-layer perceptron for fitness level classification
- `CalorieNet` — Regression network for TDEE prediction
- Full training loop with loss/accuracy curves and epoch tracking

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)

ROOT      = Path('..') if Path('../datasets').exists() else Path('.')
DATA_DIR  = ROOT / 'datasets'
MODEL_DIR = ROOT / 'ai_models' / 'dl_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch {torch.__version__}')
print(f'   Device : {device}')
print(f'   CUDA   : {torch.cuda.is_available()}')

## 2. Load & Preprocess Data

In [ ]:
df = pd.read_csv(DATA_DIR / 'fitness_profiles.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
FEATURES = ['age', 'weight_kg', 'height_cm', 'activity_level', 'gender']
TARGET_CLF = 'fitness_level'
TARGET_REG = 'calories_tdee'

le = LabelEncoder()
y_clf = le.fit_transform(df[TARGET_CLF])
y_reg = df[TARGET_REG].values.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES]).astype(np.float32)

print(f'Classes: {le.classes_}')
print(f'X shape: {X_scaled.shape}')
print(f'y_clf unique: {np.unique(y_clf)}')
print(f'y_reg range : {y_reg.min():.0f} – {y_reg.max():.0f} kcal')

## 3. PyTorch Dataset

In [ ]:
class FitnessDataset(Dataset):
    def __init__(self, X, y_clf, y_reg):
        self.X     = torch.tensor(X, dtype=torch.float32)
        self.y_clf = torch.tensor(y_clf, dtype=torch.long)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_clf[idx], self.y_reg[idx]


dataset    = FitnessDataset(X_scaled, y_clf, y_reg)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)

print(f'Train samples: {train_size}')
print(f'Val samples  : {val_size}')
print(f'Train batches: {len(train_loader)}')

## 4. Model Architecture

In [ ]:
class FitnessNet(nn.Module):
    """
    Multi-task MLP:
    - Shared encoder (feature extraction)
    - Classifier head  → fitness level (0 / 1 / 2)
    - Regression head  → TDEE calories
    """
    def __init__(self, input_dim=5, num_classes=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(),
        )
        self.clf_head = nn.Linear(64, num_classes)
        self.reg_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        features = self.encoder(x)
        return self.clf_head(features), self.reg_head(features).squeeze(-1)

model = FitnessNet(input_dim=5, num_classes=len(le.classes_)).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## 5. Training Loop

In [ ]:
# Loss functions and optimiser
clf_criterion = nn.CrossEntropyLoss()
reg_criterion = nn.MSELoss()
optimizer     = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler     = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 50
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for X_b, y_clf_b, y_reg_b in loader:
            X_b      = X_b.to(device)
            y_clf_b  = y_clf_b.to(device)
            y_reg_b  = y_reg_b.to(device)

            clf_out, reg_out = model(X_b)
            loss_clf = clf_criterion(clf_out, y_clf_b)
            loss_reg = reg_criterion(reg_out, y_reg_b) / 1e6  # scale down
            loss     = loss_clf + 0.3 * loss_reg

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(X_b)
            preds       = clf_out.argmax(dim=1)
            correct    += (preds == y_clf_b).sum().item()
            total      += len(X_b)

    return total_loss / total, correct / total

print(f'Starting training — {EPOCHS} epochs on {device}')
print('-' * 50)
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_DIR / 'fitness_net_best.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Loss: {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc: {tr_acc:.3f}/{vl_acc:.3f}')

print(f'\n✅ Best val accuracy: {best_val_acc:.4f}')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, history['train_loss'], label='Train', color='#4F86C6', linewidth=2)
ax1.plot(epochs, history['val_loss'],   label='Val',   color='#E07A5F', linewidth=2)
ax1.set_title('Loss Curve', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='#4F86C6', linewidth=2)
ax2.plot(epochs, [a*100 for a in history['val_acc']],   label='Val',   color='#E07A5F', linewidth=2)
ax2.set_title('Accuracy Curve', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('FitnessNet — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## 7. Evaluation & Confusion Matrix

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_DIR / 'fitness_net_best.pth', map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, y_clf_b, _ in val_loader:
        clf_out, _ = model(X_b.to(device))
        all_preds.extend(clf_out.argmax(dim=1).cpu().numpy())
        all_targets.extend(y_clf_b.numpy())

print('📊 Final Evaluation (Validation Set)')
print(f'   Accuracy: {accuracy_score(all_targets, all_preds):.4f}')
print()
print(classification_report(all_targets, all_preds, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('FitnessNet — Confusion Matrix (Validation)', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 8. Save Final Model

In [ ]:
# Save model weights (used by backend if loaded)
torch.save(model.state_dict(), MODEL_DIR / 'fitness_net.pth')

# Save scaler + label encoder for inference
import joblib
joblib.dump(scaler, MODEL_DIR / 'fitness_net_scaler.pkl')
joblib.dump(le,     MODEL_DIR / 'fitness_net_labels.pkl')

print('✅ Saved:')
for f in ['fitness_net.pth', 'fitness_net_best.pth',
          'fitness_net_scaler.pkl', 'fitness_net_labels.pkl']:
    path = MODEL_DIR / f
    size = path.stat().st_size / 1024 if path.exists() else 0
    print(f'   {f:<40} {size:6.1f} KB')